# Notebook 13: Checkpoints y Automatización

**Duración**: 30 minutos | **Nivel**: Avanzado

## Introducción

Los Checkpoints permiten ejecutar múltiples validaciones de forma automatizada.

### Objetivos:
1. Crear Checkpoints
2. Ejecutar validaciones batch
3. Configurar acciones automáticas
4. Integrar en pipelines

In [ ]:
import great_expectations as gx
import pandas as pd
from pathlib import Path

# Usar File Context
gx_dir = Path("../gx_project")
context = gx.get_context(mode="file", project_root_dir=str(gx_dir))

print(" Context cargado")

## Crear Checkpoint

In [ ]:
# Configurar datasource y suite si no existen
try:
    datasource = context.data_sources.get("ventas_prod")
except:
    datasource = context.data_sources.add_pandas(name="ventas_prod")
    asset = datasource.add_dataframe_asset(name="ventas")
    batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")
    
    suite = context.suites.add(gx.ExpectationSuite(name="suite_checkpoint"))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")
    )
    suite.save()

print(" Configuración lista")

## Ejecutar Validación Automatizada

In [ ]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

# Obtener componentes
datasource = context.data_sources.get("ventas_prod")
asset = datasource.get_asset("ventas")
batch_def = asset.get_batch_definition("batch_completo")
suite = context.suites.get("suite_checkpoint")

# Crear validation definition
val_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite,
        name="validacion_checkpoint"
    )
)

# Ejecutar
resultado = val_def.run(batch_parameters={"dataframe": df})

print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")
print(f"Expectativas evaluadas: {len(resultado.results)}")

## Script de Automatización

In [ ]:
# Crear script Python para automatización
script_content = '''#!/usr/bin/env python
"""Script de validación automatizada"""
import great_expectations as gx
import pandas as pd
import sys
from pathlib import Path

def validar_datos(archivo_csv):
    # Cargar contexto
    context = gx.get_context(mode="file", project_root_dir="./gx_project")
    
    # Cargar datos
    df = pd.read_csv(archivo_csv)
    
    # Ejecutar validación
    datasource = context.data_sources.get("ventas_prod")
    asset = datasource.get_asset("ventas")
    batch_def = asset.get_batch_definition("batch_completo")
    suite = context.suites.get("suite_checkpoint")
    
    val_def = context.validation_definitions.get("validacion_checkpoint")
    resultado = val_def.run(batch_parameters={"dataframe": df})
    
    # Generar Data Docs
    context.build_data_docs()
    
    # Retornar código de salida
    return 0 if resultado.success else 1

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Uso: python validar.py <archivo.csv>")
        sys.exit(1)
    
    exit_code = validar_datos(sys.argv[1])
    sys.exit(exit_code)
'''

# Guardar script
with open("../validar.py", "w") as f:
    f.write(script_content)

print(" Script de automatización creado: validar.py")
print("\nUso: python validar.py data/ventas_sucias.csv")

##  Ejercicio

Modifica el script para enviar un email cuando la validación falla.

In [ ]:
# TU CÓDIGO AQUÍ
pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  Checkpoints automatizan validaciones
2.  Scripts Python para integración
3.  Códigos de salida para CI/CD
4.  Acciones automáticas configurables

**Próximo**: Notebook 14 - CI/CD 